In [12]:
import json
import re

In [3]:
microservices = ["globeco-allocation-service",
"globeco-confirmation-service",
"globeco-execution-service",
"globeco-fix-engine",
"globeco-order-generation-service",
"globeco-order-service",
"globeco-portfolio-accounting-service",
"globeco-portfolio-management-portal",
"globeco-portfolio-service",
"globeco-pricing-service",
"globeco-security-service",
"globeco-trade-service",]
len(microservices)

12

In [5]:
cpu_usage_file = "../data/cpu_usage.json"
with open(cpu_usage_file) as f:
    cpu_usage = json.load(f)

cpu_usage

{'envoy': {'mean': 0.0258679409,
  'std': 0.0067618232,
  'min': 0.0027068092,
  'max': 0.0378731285},
 'envoy-gateway': {'mean': 0.001456258,
  'std': 0.0003830543,
  'min': 0.0005482,
  'max': 0.0035188937},
 'globeco-allocation-service': {'mean': 0.0009103815,
  'std': 0.0003014673,
  'min': 0.0001017333,
  'max': 0.0018225759},
 'globeco-allocation-service-postgresql': {'mean': 0.0047128867,
  'std': 0.0007316236,
  'min': 0.0016634333,
  'max': 0.0064288075},
 'globeco-confirmation-service': {'mean': 0.0116967887,
  'std': 0.0030849664,
  'min': 0.0030440165,
  'max': 0.0182568226},
 'globeco-debug-tools': {'mean': 0.0, 'std': 0.0, 'min': 0.0, 'max': 0.0},
 'globeco-execution-service': {'mean': 0.0803571761,
  'std': 0.0308216057,
  'min': 0.0019400979,
  'max': 0.3445125381},
 'globeco-execution-service-postgresql': {'mean': 0.0145068524,
  'std': 0.0036297467,
  'min': 0.0013685667,
  'max': 0.0240708715},
 'globeco-fix-engine': {'mean': 0.0174847617,
  'std': 0.0071653686,
  'm

In [19]:
memory_usage_file = "../data/memory_usage.json"
with open(memory_usage_file) as f:
    memory_usage = json.load(f)

memory_usage

{'envoy': {'mean': 30.7177815083,
  'std': 1.5713525561,
  'min': 20.921875,
  'max': 31.953125},
 'envoy-gateway': {'mean': 51.5346397211,
  'std': 1.1410176035,
  'min': 46.60546875,
  'max': 53.5234375},
 'globeco-allocation-service': {'mean': 15.8049872713,
  'std': 2.1570409134,
  'min': 9.80078125,
  'max': 18.30859375},
 'globeco-allocation-service-postgresql': {'mean': 107.4919384593,
  'std': 0.8184618648,
  'min': 104.13671875,
  'max': 109.59765625},
 'globeco-confirmation-service': {'mean': 26.6174411526,
  'std': 5.5826518802,
  'min': 3.7265625,
  'max': 34.9296875},
 'globeco-debug-tools': {'mean': 4.1774553571,
  'std': 0.3419093588,
  'min': 3.734375,
  'max': 4.69140625},
 'globeco-execution-service': {'mean': 272.0740896178,
  'std': 11.4178399012,
  'min': 208.0390625,
  'max': 281.16796875},
 'globeco-execution-service-postgresql': {'mean': 141.0458004354,
  'std': 14.4785018966,
  'min': 96.98046875,
  'max': 162.01171875},
 'globeco-fix-engine': {'mean': 17.96702

In [6]:
memory = {'globeco-allocation-service': 100,
 'globeco-confirmation-service': 100,
 'globeco-execution-service': 700,
 'globeco-fix-engine': 100,
 'globeco-order-generation-service': 700,
 'globeco-order-service': 700,
 'globeco-portfolio-accounting-service': 100,
 'globeco-portfolio-management-portal': 200,
 'globeco-portfolio-service': 300,
 'globeco-pricing-service': 1000,
 'globeco-security-service': 200,
 'globeco-trade-service': 700}

In [7]:
cpu_limit = {'globeco-allocation-service': 200,
 'globeco-confirmation-service': 200,
 'globeco-execution-service': 720,
 'globeco-fix-engine': 200,
 'globeco-order-generation-service': 1000,
 'globeco-order-service': 1000,
 'globeco-portfolio-accounting-service': 200,
 'globeco-portfolio-management-portal': 587,
 'globeco-portfolio-service': 520,
 'globeco-pricing-service': 1000,
 'globeco-security-service': 425,
 'globeco-trade-service': 1000}

In [23]:
# {{/*
# Selector labels
# */}}
# {{- define "globeco.selectorLabels" -}}
# app.kubernetes.io/name: {{ include "globeco.name" . }}
# app.kubernetes.io/instance: {{ .Release.Name }}
# {{- end }}

def to_camel_case(text: str) -> str:
    # Replace hyphens and underscores with spaces, then split into words
    words = re.sub(r'[-_]+', ' ', text).split()
    
    if not words:
        return ""
    
    # Lowercase the first word, capitalize the rest, and join them
    return words[0].lower() + "".join(word.capitalize() for word in words[1:])



with open("../globeco/templates/_resources.tpl", "w", encoding="utf-8") as tpl:
    for microservice in microservices:
        # convert microservice name to camel case
        microservice_cc = to_camel_case(microservice)
        
        # generate the template comment for the microservice
        tpl.write("{{/*\n")
        tpl.write(f"Resource allocation for {microservice}\n")
        tpl.write("*/}}\n")
        
        # generate the define statment for the microservice resource
        tpl.write("{{- ") 
        tpl.write(f'define "globeco.{microservice_cc}Resources"')
        tpl.write(" -}}\n")
        
        # calculate the cpu allocation request
        max_cpu = cpu_usage[microservice]["max"]
        max_cpu_millicores = max(round(max_cpu * 1000/0.70), 50)
        
        # calculate the memory allocatin request
        max_mem_mi = memory_usage[microservice]["max"]/0.70
        # round up to the nearest 100
        max_mem_mi = int(max_mem_mi) + (100 - int(max_mem_mi) % 100)
        # print(f"{microservice}: {max_cpu_millicores}m")
        # print()
        tabs = " " * 10
        tabs = ""
        tpl.write(tabs + "resources:\n")
        tpl.write(tabs + "  requests:\n")
        for i in range(2,7):
            cpu_request = max_cpu_millicores * ((i+1) * 0.5)
            mem_request = max_mem_mi * ((i+1) * 0.5)
            if i == 2:
                tpl.write(tabs + "    {{- if eq .Values.autoscaler \"vertical-" + str(i) + "\" }}\n")    
            else:
                tpl.write(tabs + "    {{- else if eq .Values.autoscaler \"vertical-" + str(i) + "\" }}\n")    
            tpl.write(tabs + f"    cpu: \"{int(cpu_request)}m\"\n")
            tpl.write(tabs + f"    memory: \"{int(mem_request)}Mi\"\n")
        
        tpl.write(tabs + "    {{- else }}\n")
        tpl.write(tabs + f"    cpu: \"{int(max_cpu_millicores)}m\"\n")
        tpl.write(tabs + f"    memory: \"{int(max_mem_mi)}Mi\"\n")
        tpl.write(tabs + "    {{- end }}\n")
        tpl.write(tabs + "  limits:\n")
        # print(tabs + f"    memory: \"{mem_request}Mi\"")
        tpl.write(tabs + f"    memory: \"6000Mi\"\n")
        tpl.write("{{- end }}\n\n") 
        
        # print()
        # print()



